# Saving Brains — KMC-400-20y
## M9: TAP K=2 Predictive Model — Binary Cognitive Phenotype
### Phase 1 (Birth → 3 months CA) · Phase 2 (Birth → 12 months CA)

**Target:** TAP clustering with K=2 — GO-1 Attentive vs GO-2 Inattentive
**Task:** Binary classification (GO-2 Inattentive (risk))

---

### Key differences from M8 (WASI K=3)

| Aspect | M8 — WASI K=3 | M9 — TAP K=2 |
|--------|---------------|-----------------|
| Cluster source | `clusters_GO_WASI_k3.csv` | `clusters_GO_TAP_revised_v1.csv` |
| Classes | 3 (High/Medium/Low IQ) | **2** (GO-1 Attentive / GO-2 Inattentive) |
| XGBoost objective | `multi:softprob` | **`binary:logistic`** |
| Primary metric | Macro-AUC | AUC-ROC (binary) |
| Multiclass section | Yes (3×3 confusion) | **Not needed** |
| Binary collapse | GO-3 vs rest | **Direct binary** |
| scale_pos_weight | via sample_weight | **SPW = n_GO1 / n_GO2** |

### Clinical rationale
TAP phenotype (attention/WM) is clinically orthogonal to
# the WASI+CVLT composite GO-i. Compare with M7 as reference only.
# Lower TAP WM omissions → better attention → lower risk.


## 1. Setup

In [0]:
# Databricks only: uncomment to restart Python after pip installs
# dbutils.library.restartPython()
print("Environment ready.")

In [0]:
# Install required packages (uncomment if not already installed)
# pip install "numpy<2" "xgboost<3" shap statsmodels mlflow --quiet
import sys
print("Python:", sys.version)

In [0]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    brier_score_loss, roc_curve, confusion_matrix,
)
from xgboost import XGBClassifier
import shap
import joblib

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_theme(style="whitegrid")

import mlflow
import mlflow.xgboost

# ── Paths — adjust to your environment ───────────────────────────────────
DATA_MASTER  = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
DATA_CLUSTER = "/Volumes/workspace/default/kmc_data/clusters_GO_WASI_k3.csv"
EXP_NAME     = "/Users/a.echeverrig@uniandes.edu.co/prediccion-ci-prematuros-madre-canguro/notebooks/SavingBrains_M7_TwoPhase"

mlflow.set_registry_uri("databricks")
mlflow.set_experiment(EXP_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment         :", EXP_NAME)
print("xgboost:", __import__('xgboost').__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


## 2. Constants, Criteria & Feature Groups

In [0]:
RANDOM_STATE = 42
N_FOLDS      = 5

# ─────────────────────────────────────────────────────────────────────────
# EXPLICIT INCLUSION CRITERIA
# (expert requirement: make criteria explicit and replicable)
# ─────────────────────────────────────────────────────────────────────────
INCLUSION_CRITERIA = {
    "temporal"      : "Measured BEFORE the outcome window (before 20 years)",
    "codebook"      : "Confirmed in SPSS codebook (Feb 2024) OR expert-validated",
    "coverage"      : "Available in >= 60% of G1 preterm sample (n_valid >= 232)",
    "non_leakage"   : "Not a proxy of the cognitive outcome (GO-i) itself",
    "collinearity"  : "Pearson r < 0.85 with all other features in the same model",
    "justification" : "Has a published clinical or bibliographic rationale",
}

EXCLUSION_CRITERIA = {
    "outcome_var"   : "Measured at 20-year follow-up (WASI_, CVLT_, TAP_, NHPT_, etc.)",
    "codebook_miss" : "Not in codebook AND not expert-confirmed",
    "low_coverage"  : "Available in < 60% of sample",
    "same_construct": "Redundant with another included feature (same construct)",
    "high_corr"     : "r >= 0.85 with another feature already in model",
}

print("Inclusion criteria:")
for k, v in INCLUSION_CRITERIA.items():
    print(f"  [{k}] {v}")
print()
print("Exclusion criteria:")
for k, v in EXCLUSION_CRITERIA.items():
    print(f"  [{k}] {v}")

# ─────────────────────────────────────────────────────────────────────────
# FENTON 2013 HC REFERENCE TABLES
# Source: Fenton TR & Kim JH (2013) BMC Pediatrics 13:59
# Formula: z = (X / M - 1) / S   [LMS method, L≈1, Cole & Green 1992]
# Valid range: GA 22–41 weeks (birth and 40-week corrected age ONLY)
# Beyond 40w CA: WHO curves with corrected age (NUT12M_* variables)
# ─────────────────────────────────────────────────────────────────────────
FENTON_HC_M = {
    24:21.8, 25:23.0, 26:24.2, 27:25.4, 28:26.6, 29:27.8, 30:28.9,
    31:30.0, 32:31.0, 33:31.9, 34:32.7, 35:33.4, 36:34.1, 37:34.7,
    38:35.1, 39:35.5, 40:35.9, 41:36.1,
}
FENTON_HC_S = {
    24:.058, 25:.056, 26:.054, 27:.052, 28:.050, 29:.048, 30:.046,
    31:.045, 32:.044, 33:.043, 34:.042, 35:.041, 36:.041, 37:.040,
    38:.040, 39:.040, 40:.040, 41:.040,
}

# ─────────────────────────────────────────────────────────────────────────
# FEATURE GROUPS — PHASE 1 (Birth → 3 months corrected age)
# ─────────────────────────────────────────────────────────────────────────

# F1A: KMC intervention
# Special imputation: TC group = 0 (no kangaroo care by study design)
FEATURES_P1_KMC = [
    "EX41_durPCconcontroles03",     # KMC duration (hours) — TC imputed as 0 ***expert
]

# F1B: Neonatal severity (confirmed in codebook)
FEATURES_P1_NEONATAL_CB = [
    "NEO_ventilacion",              # Mechanical ventilation type  **
    "NEO_UCI",                      # NICU admission  *
    "NEO_diasaminoglucosidos",      # Days of aminoglycosides  **  (ototoxic: auditory risk)
    "NEO_recibioaminiglucosidos",   # Received aminoglycosides (binary)  †
]

# F1C: Neonatal severity (in dataset, expert-confirmed — not in codebook)
FEATURES_P1_NEONATAL_DS = [
    "NEO_fotote6",                  # Days of phototherapy  ***
    "NEO_HOSP",                     # Days of hospitalisation  **
    "NEO_diasventiladores",         # Days of mechanical ventilation  *
    "NEO_totoxidias",               # Days of oxygen therapy  **
    "NEO_parentNutritiondias",      # Days of parenteral nutrition  **
]

# F1D: Early neurological screening (neonatal)
# Note: APGAR p=0.38 ns in bivariate but included per expert guidance
#       as early warning signal — XGBoost may find joint effects
FEATURES_P1_NEURO = [
    "BIRTH_apgar1_5",           # APGAR score at 1 min  ns (expert: early alert)
    "BIRTH_apgar5_5",           # APGAR score at 5 min  ns (expert: early alert)
    "NEURO_leucomalaciacat",    # Periventricular leukomalacia  *  (58% data)
]

# F1E: Fenton growth — birth and 40 weeks (derived; source vars in codebook)
# F_catchup_hc_fenton and F_z_hc_birth_fenton — built in Section 4
FEATURES_P1_FENTON = [
    "F_catchup_hc_fenton",      # HC z-score change birth→40w  ***
    "F_z_hc_birth_fenton",      # HC z-score at birth  *
]

# F1F: Growth velocity (WHO z-scores, corrected age) — 40w→3m CA
# Velocity = delta z-score; avoids multicollinearity vs levels (r<0.46)
# Source: NUT12M_wam8 (41w), NUT12M_wam9 (3m CA), NIHS reference
FEATURES_P1_VELOCITY = [
    "F_delta_waz_40w_3m",       # Δ weight-for-age z 40w→3m  ns
    "F_delta_haz_40w_3m",       # Δ height-for-age z 40w→3m  ns
    "F_delta_hcz_40w_3m",       # Δ HC-for-age z 40w→3m  ns
    "NUT12M_pcam5",             # HC/expected at birth (WHO ratio ×100)  *
]

# F1G: Breastfeeding
FEATURES_P1_FEEDING = [
    "NUT12M_LM12m",             # Breastfeeding status  ns (clinical relevance)
]

# F1H: Socioeconomic context
FEATURES_P1_SES = [
    "SCB_nivm1",                # Maternal education level  **
    "SCB_nivp1",                # Paternal education level  *  (new)
    "SCB_percap1",              # Household per-capita income  ns
]

# All Phase 1 features
ALL_FEATURES_P1 = (
    FEATURES_P1_KMC +
    FEATURES_P1_NEONATAL_CB +
    FEATURES_P1_NEONATAL_DS +
    FEATURES_P1_NEURO +
    FEATURES_P1_FENTON +
    FEATURES_P1_VELOCITY +
    FEATURES_P1_FEEDING +
    FEATURES_P1_SES
)

# ─────────────────────────────────────────────────────────────────────────
# FEATURE GROUPS — PHASE 2 (adds 3m → 12m CA variables)
# Includes all Phase 1 features + the following
# ─────────────────────────────────────────────────────────────────────────

# F2A: Psychomotor development — Griffiths (6m and 12m CA)
FEATURES_P2_GRIFFITHS = [
    "PMD_RSM6",      # Raw motor score 6m  ***
    "PMD_coaudl6",   # Auditory quotient 6m  ***
    "PMD_cogrif6",   # General quotient 6m  **
    "PMD_coloco6",   # Locomotion 6m  *
    "PMD_cogrif12",  # General quotient 12m  *
    "PMD_coloco12",  # Locomotion 12m  *
    "PMD_copers12",  # Personal-social 12m  *
]

# F2B: Growth velocity — 3m→12m CA (expert: "how I got there matters")
FEATURES_P2_VELOCITY = [
    "F_delta_waz_3m_12m",   # Δ WAZ 3m→12m  *  (GO-2 shows paradoxical catch-up)
    "F_delta_haz_3m_12m",   # Δ HAZ 3m→12m  *
    "F_delta_hcz_3m_12m",   # Δ HCZ 3m→12m  ns
]

# F2C: Home environment (12m CA)
FEATURES_P2_HOME = [
    "HOMET_tsubs_tothome12mfact",     # HOME factor score  *
    "HOMET_ttotalsubscalehome12mraw", # HOME total raw  †
]

# F2D: Neurodevelopmental screening (6–12m CA)
# INFANIB: p=ns in bivariate but expert-recommended
FEATURES_P2_NEURODEV = [
    "FOLL12M_infanib9",    # INFANIB 3m CA  ns (expert: key clinical exam)
    "FOLL12M_infanib12",   # INFANIB 12m CA  ns
]

# F2E: Condition at birth (retained in Phase 2)
FEATURES_P2_BIRTH = [
    "BIRTH_peso5",   # Birth weight (g)  ns
    "EX41_talla8",   # Length at 40w CA (mm)  †
    "EX41_peso8",    # Weight at 40w CA (g)  ns
]

# All Phase 2 features = Phase 1 + additional
ALL_FEATURES_P2 = (
    ALL_FEATURES_P1 +
    FEATURES_P2_GRIFFITHS +
    FEATURES_P2_VELOCITY +
    FEATURES_P2_HOME +
    FEATURES_P2_NEURODEV +
    FEATURES_P2_BIRTH
)

# ── XGBoost hyperparameters ───────────────────────────────────────────────
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# ── Colour palette ────────────────────────────────────────────────────────
PAL = {
    "dark_blue"  : "#1A5276",
    "mid_blue"   : "#2E86C1",
    "light_blue" : "#AED6F1",
    "red"        : "#C0392B",
    "green"      : "#1E8449",
    "orange"     : "#D35400",
    "purple"     : "#884EA0",
}

print("Phase 1 features:", len(ALL_FEATURES_P1))
print("Phase 2 features:", len(ALL_FEATURES_P2))
print("Added in Phase 2:", len(ALL_FEATURES_P2) - len(ALL_FEATURES_P1))


In [0]:
# ─────────────────────────────────────────────────────────────────────────
# TAP K=2 — XGBoost binary configurations
#
# K=2 means pure binary: no multiclass, no softprob, no collapse needed.
# # TAP K=2 — pure binary classification
# No multiclass needed: GO-1 Attentive vs GO-2 Inattentive
# scale_pos_weight accounts for mild class imbalance
# ─────────────────────────────────────────────────────────────────────────

# Baseline binary
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    n_estimators     = 200,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# Anti-overfitting
XGB_PARAMS_REG = dict(
    objective        = "binary:logistic",
    n_estimators     = 150,
    learning_rate    = 0.03,
    max_depth        = 3,
    subsample        = 0.7,
    colsample_bytree = 0.7,
    min_child_weight = 8,
    reg_lambda       = 4.0,
    reg_alpha        = 0.5,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# scale_pos_weight — computed after loading data
# SPW = n_GO1 / n_GO2 (approximate: 200/183)
# Filled in after load_cohort() below
XGB_PARAMS_SPW = dict(
    **XGB_PARAMS_REG,
    scale_pos_weight = 1.0,   # updated dynamically after data load
)

# Class labels
CLASS_LABELS = {1: "GO-1 Attentive", 2: "GO-2 Inattentive"}

print("Binary XGBoost configs:")
print("  XGB_PARAMS     : baseline binary")
print("  XGB_PARAMS_REG : anti-overfitting (L2=4, mcw=8, subsample=0.7)")
print("  XGB_PARAMS_SPW : + scale_pos_weight (computed after data load)")
print("  CLASS_LABELS  :", CLASS_LABELS)


## 3. Data Loading — TAP K=2 Clusters

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Data loading for TAP K=2 clustering
#
# TAP K=2 label mapping:
#   CSV value 1 → GO-1 Attentive (GO-1, lower risk)
#   CSV value 2 → GO-2 Inattentive (GO-2, higher risk)
#   y = 1 if GO-2 (risk), 0 if GO-1
# ─────────────────────────────────────────────────────────────────────────

DATA_MASTER  = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
DATA_CLUSTER = "/Volumes/workspace/default/kmc_data/clusters_GO_TAP_revised_v1.csv"
MLFLOW_URI   = "mlruns"
EXP_NAME     = "SavingBrains_M9_TAP_K2"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXP_NAME)


def load_cohort_k2(master_path: str, cluster_path: str,
                   risk_class: int = 2) -> pd.DataFrame:
    """
    Load master dataset and merge TAP K=2 cluster labels.

    Label mapping
    -------------
    CSV value 1 → GO-1 Attentive → y = 0 (not at risk)
    CSV value 2 → GO-2 Inattentive → y = 1 (at risk)

    XGBoost binary: y in {0, 1}.
    scale_pos_weight = n_y0 / n_y1 is computed and returned.
    """
    df = pd.read_csv(master_path, low_memory=False)
    cl = pd.read_csv(cluster_path)[["code", "GO_i"]]
    df = df.merge(cl, on="code", how="inner")

    to_n  = lambda s: pd.to_numeric(s, errors="coerce")
    grupo = to_n(df["ubicac"])
    pret  = to_n(df["preterm"])
    mask  = grupo.isin([1, 2]) & (pret == 1) & df["GO_i"].notna()
    df_out = df[mask].copy().reset_index(drop=True)

    df_out["GO_i_orig"] = df_out["GO_i"].astype(int)
    # y = 1 if risk class (GO-2), 0 otherwise
    df_out["y"] = (df_out["GO_i_orig"] == risk_class).astype(int)

    return df_out


df_g1    = load_cohort_k2(DATA_MASTER, DATA_CLUSTER, risk_class=2)
grupo_g1 = pd.to_numeric(df_g1["ubicac"], errors="coerce")
y_arr    = df_g1["y"].values

n_total = len(df_g1)
n_risk  = int(y_arr.sum())
n_safe  = int((y_arr == 0).sum())
spw_val = round(n_safe / n_risk, 4)

# Update SPW in XGB_PARAMS_SPW dynamically
XGB_PARAMS_SPW["scale_pos_weight"] = spw_val

print("TAP K=2 cohort loaded:")
print(f"  Total           : {n_total}")
print(f"  GO-1 Attentive        : n={n_safe} ({100*n_safe/n_total:.1f}%)  → y=0")
print(f"  GO-2 Inattentive      : n={n_risk} ({100*n_risk/n_total:.1f}%)  → y=1")
print(f"  Prevalence      : {y_arr.mean():.3f}")
print(f"  scale_pos_weight: {spw_val} (XGB_PARAMS_SPW updated)")


## 4. Feature Engineering

In [0]:
def fenton_z(value_cm: float, ga_weeks: float,
             M_table: dict, S_table: dict) -> float:
    """
    Fenton 2013 HC z-score using LMS method (L≈1).
    z = (X / M - 1) / S
    Valid for GA 22–41 weeks only.
    """
    if pd.isna(value_cm) or pd.isna(ga_weeks):
        return np.nan
    ga_int = int(round(ga_weeks))
    if ga_int not in M_table:
        return np.nan
    return (value_cm / M_table[ga_int] - 1) / S_table[ga_int]


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive all engineered features and apply special imputations.

    Special imputation rules
    ------------------------
    EX41_durPCconcontroles03 (KMC duration):
        TC group (ubicac==2) → 0.0 by study design.
        TC patients did not receive kangaroo care; their NaN values
        represent structural zeros, not missing data.
        KMC group: raw values retained.

    Fenton HC z-scores
    ------------------
    F_z_hc_birth_fenton   : HC z-score at birth (GA-adjusted, Fenton 2013)
    F_z_hc_40w_fenton     : HC z-score at 40 corrected weeks
    F_catchup_hc_fenton   : z_40w - z_birth  (neonatal brain growth proxy)
                            > 0 = improved position vs GA peers
                            < 0 = lost position vs GA peers (risk)

    Growth velocity (WHO z-scores, corrected age — NIHS reference)
    ---------------------------------------------------------------
    Using velocity (delta z-score) instead of level z-scores avoids
    multicollinearity (r=0.56-0.88 between longitudinal levels).

    F_delta_waz_40w_3m  : Δ weight-for-age z  40w→3m CA
    F_delta_haz_40w_3m  : Δ height-for-age z  40w→3m CA
    F_delta_hcz_40w_3m  : Δ HC-for-age z      40w→3m CA
    F_delta_waz_3m_12m  : Δ weight-for-age z  3m→12m CA  (* p=0.029)
    F_delta_haz_3m_12m  : Δ height-for-age z  3m→12m CA  (* p=0.046)
    F_delta_hcz_3m_12m  : Δ HC-for-age z      3m→12m CA
    """
    df = df.copy()
    to_n = lambda s: pd.to_numeric(s, errors="coerce")
    grupo = to_n(df["ubicac"])

    # ── KMC duration: TC group → 0 by design ─────────────────────────────
    kmc_raw = to_n(df["EX41_durPCconcontroles03"])
    df["EX41_durPCconcontroles03"] = np.where(grupo == 2, 0.0, kmc_raw)

    # ── Fenton HC z-scores ────────────────────────────────────────────────
    ga_birth = to_n(df["BIRTH_ballar5"])
    hc_birth = to_n(df["BIRTH_pc5"]) / 10.0   # mm -> cm
    hc_40w   = to_n(df["EX41_pc8"])            # already cm

    df["F_z_hc_birth_fenton"] = [
        fenton_z(v, e, FENTON_HC_M, FENTON_HC_S)
        for v, e in zip(hc_birth, ga_birth)
    ]
    df["F_z_hc_40w_fenton"] = [
        fenton_z(v, 40, FENTON_HC_M, FENTON_HC_S)
        for v in hc_40w
    ]
    df["F_catchup_hc_fenton"] = (
        df["F_z_hc_40w_fenton"] - df["F_z_hc_birth_fenton"]
    )

    # ── Growth velocity (WHO z-scores with corrected age) ─────────────────
    # Source: NUT12M_wam/ham/pcam at 41w, 3m, 12m CA (NIHS reference)
    wam8  = to_n(df["NUT12M_wam8"])    # WAZ at 41 weeks
    wam9  = to_n(df["NUT12M_wam9"])    # WAZ at 3m CA
    wam12 = to_n(df["NUT12M_wam12"])   # WAZ at 12m CA
    ham8  = to_n(df["NUT12M_ham8"])    # HAZ at 41 weeks
    ham9  = to_n(df["NUT12M_ham9"])    # HAZ at 3m CA
    ham12 = to_n(df["NUT12M_ham12"])   # HAZ at 12m CA
    pcam8 = to_n(df["NUT12M_pcam8"])   # HCZ at 41 weeks
    pcam9 = to_n(df["NUT12M_pcam9"])   # HCZ at 3m CA
    pcam12= to_n(df["NUT12M_pcam12"])  # HCZ at 12m CA

    df["F_delta_waz_40w_3m"]  = wam9  - wam8
    df["F_delta_haz_40w_3m"]  = ham9  - ham8
    df["F_delta_hcz_40w_3m"]  = pcam9 - pcam8
    df["F_delta_waz_3m_12m"]  = wam12 - wam9
    df["F_delta_haz_3m_12m"]  = ham12 - ham9
    df["F_delta_hcz_3m_12m"]  = pcam12 - pcam9

    return df


df_g1 = build_features(df_g1)

# ── Validation ────────────────────────────────────────────────────────────
print("KMC duration imputation check:")
to_n = lambda s: pd.to_numeric(s, errors="coerce")
grupo_g1 = to_n(df_g1["ubicac"])
kmc_col  = to_n(df_g1["EX41_durPCconcontroles03"])
print("  KMC group mean :", round(kmc_col[grupo_g1==1].mean(), 2))
print("  TC  group mean :", round(kmc_col[grupo_g1==2].mean(), 2), "(should be 0.0)")
print("  TC  group NaN  :", int(kmc_col[grupo_g1==2].isna().sum()), "(should be 0)")

print()
print("Fenton features:")
go = df_g1["GO_i"]
for col in ["F_catchup_hc_fenton", "F_z_hc_birth_fenton"]:
    pct = df_g1[col].notna().mean() * 100
    m1  = df_g1.loc[go==1, col].mean()
    m2  = df_g1.loc[go==2, col].mean()
    print(f"  {col:<30} {pct:.0f}% | GO-1={m1:.3f}  GO-2={m2:.3f}")

print()
print("Velocity features (multicollinearity check — expect r < 0.50):")
vel_cols = ["F_delta_waz_40w_3m", "F_delta_waz_3m_12m",
            "F_delta_haz_40w_3m", "F_delta_haz_3m_12m"]
print(df_g1[vel_cols].corr().round(2).to_string())


## 5. Univariate Screening — Mann-Whitney U (binary)

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Mann-Whitney U (binary target: GO-1 Attentive vs GO-2 Inattentive)
# Simpler than WASI K=3: no Kruskal-Wallis needed — direct Mann-Whitney U
# ─────────────────────────────────────────────────────────────────────────

def univariate_screen_k2(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """
    Mann-Whitney U screening for binary target (TAP K=2).

    Tests
    -----
    Mann-Whitney U (two-sided): GO-1 GO-1 Attentive vs GO-2 GO-2 Inattentive.
    FDR correction: Benjamini-Hochberg on raw p-values.
    """
    rows = []
    y_col = df["y"]

    for col in feature_cols:
        if col not in df.columns:
            continue
        s   = pd.to_numeric(df[col], errors="coerce")
        pct = s.notna().mean() * 100
        s0  = s[y_col == 0].dropna()   # GO-1
        s1  = s[y_col == 1].dropna()   # GO-2

        try:
            _, p = mannwhitneyu(s0, s1, alternative="two-sided")
        except Exception:
            p = np.nan

        rows.append({
            "feature"  : col,
            "pct_data" : round(pct, 1),
            "mean_go1" : round(s0.mean(), 3),
            "mean_go2" : round(s1.mean(), 3),
            "delta"    : round(s1.mean() - s0.mean(), 3),
            "p_raw"    : round(p, 6) if pd.notna(p) else np.nan,
        })

    df_out = pd.DataFrame(rows)
    _, qs, _, _ = multipletests(df_out["p_raw"].fillna(1), method="fdr_bh")
    df_out["q_fdr"] = qs.round(4)
    df_out["sig"] = df_out["p_raw"].apply(
        lambda p: "***" if p < 0.001 else "**" if p < 0.01 else
                  "*" if p < 0.05 else "†" if p < 0.10 else "ns"
    )
    return df_out.sort_values("p_raw").reset_index(drop=True)


all_candidates = list(dict.fromkeys(ALL_FEATURES_P1 + ALL_FEATURES_P2))
df_screen = univariate_screen_k2(df_g1, all_candidates)

print("Univariate screening — TAP K=2 (GO-1 Attentive vs GO-2 Inattentive)")
sep = "-" * 75
print(sep)
print(df_screen[["feature","pct_data","mean_go1","mean_go2","delta","p_raw","sig"]]
      .to_string(index=False))
print(sep)
print("Significant p<0.05:", (df_screen["p_raw"] < 0.05).sum())


## 6. Core ML Functions — Binary Classification

In [0]:
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    brier_score_loss, roc_curve, confusion_matrix,
    average_precision_score,
)


def prepare_arrays(df: pd.DataFrame, feature_cols: list) -> tuple:
    """Extract X and binary y (GO-2 = 1, GO-1 = 0)."""
    to_n = lambda s: pd.to_numeric(s, errors="coerce")
    cols = [c for c in feature_cols if c in df.columns]
    X    = df[cols].apply(to_n).values.astype(float)
    y    = df["y"].values
    return X, y, cols


def cross_validate(
    X: np.ndarray, y: np.ndarray,
    params: dict, n_folds: int = 5, random_state: int = 42,
) -> dict:
    """
    Stratified K-Fold CV — binary classification.
    In-fold imputation + scaling. F1-optimal threshold.
    Returns OOF probs + full metrics dict.
    """
    skf       = StratifiedKFold(n_folds, shuffle=True, random_state=random_state)
    oof       = np.zeros(len(y))
    train_aucs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr       = y[tr_idx]

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        sc   = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)

        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        oof[te_idx] = clf.predict_proba(X_te)[:, 1]
        train_aucs.append(roc_auc_score(y_tr, clf.predict_proba(X_tr)[:, 1]))

    auc_oof  = roc_auc_score(y, oof)
    pr_auc   = average_precision_score(y, oof)
    gap      = float(np.mean(train_aucs)) - auc_oof

    thresholds = np.linspace(0.05, 0.95, 181)
    f1_scores  = [f1_score(y, (oof >= t).astype(int), zero_division=0)
                  for t in thresholds]
    best_thr = float(thresholds[np.argmax(f1_scores)])
    pred     = (oof >= best_thr).astype(int)
    cm_out   = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm_out[0,0], cm_out[0,1], cm_out[1,0], cm_out[1,1]

    metrics = {
        "auc_oof"       : float(auc_oof),
        "pr_auc"        : float(pr_auc),
        "auc_train"     : float(np.mean(train_aucs)),
        "gap"           : float(gap),
        "auc_cv_std"    : float(np.std(train_aucs)),
        "f1"            : float(f1_score(y, pred, zero_division=0)),
        "sensitivity"   : float(recall_score(y, pred, zero_division=0)),
        "specificity"   : float(recall_score(1-y, 1-pred, zero_division=0)),
        "precision_ppv" : float(precision_score(y, pred, zero_division=0)),
        "npv"           : float(tn/(tn+fn)) if (tn+fn) > 0 else float("nan"),
        "brier"         : float(brier_score_loss(y, oof)),
        "best_threshold": best_thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "fn_rate": float(fn/(tp+fn)) if (tp+fn) > 0 else float("nan"),
        "fp_rate": float(fp/(fp+tn)) if (fp+tn) > 0 else float("nan"),
    }
    return {"oof_probs": oof, "metrics": metrics, "train_aucs": train_aucs}


def fit_final_model(X: np.ndarray, y: np.ndarray, params: dict) -> dict:
    """Fit on full dataset for SHAP and serialisation."""
    imp   = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X)
    sc    = StandardScaler()
    X_sc  = sc.fit_transform(X_imp)
    clf   = XGBClassifier(**params)
    clf.fit(X_sc, y)
    return {"clf": clf, "imputer": imp, "scaler": sc, "X_scaled": X_sc}


print("Functions ready: prepare_arrays, cross_validate, fit_final_model")


## 7. MLflow Experiment Runner

In [0]:
def run_experiment(
    run_name: str, feature_cols: list, df: pd.DataFrame,
    xgb_params: dict, tags: dict = None,
    n_folds: int = 5, random_state: int = 42,
) -> dict:
    """MLflow run — TAP K=2 binary model."""
    X, y, used_cols = prepare_arrays(df, feature_cols)
    if len(used_cols) < len(feature_cols):
        print("  WARNING — features not found:", set(feature_cols)-set(used_cols))

    with mlflow.start_run(run_name=run_name) as run:
        default_tags = {
            "project"    : "SavingBrains",
            "dataset"    : "KMC-400-20y",
            "model"      : "M9_TAP_K2",
            "clustering" : "TAP_K2",
            "target"     : "GO-2 Inattentive = 1",
            "prevalence" : f"{y.mean():.3f}",
        }
        if tags:
            default_tags.update(tags)
        mlflow.set_tags(default_tags)

        mlflow.log_params({
            "n_features"  : len(used_cols),
            "n_folds"     : n_folds,
            "random_state": random_state,
            **{"xgb_" + k: v for k, v in xgb_params.items()},
        })

        cv_out = cross_validate(X, y, xgb_params, n_folds, random_state)
        m      = cv_out["metrics"]

        mlflow.log_metrics({
            "auc_oof"       : round(m["auc_oof"], 4),
            "pr_auc"        : round(m["pr_auc"], 4),
            "auc_train"     : round(m["auc_train"], 4),
            "gap"           : round(m["gap"], 4),
            "f1"            : round(m["f1"], 4),
            "sensitivity"   : round(m["sensitivity"], 4),
            "specificity"   : round(m["specificity"], 4),
            "brier"         : round(m["brier"], 4),
            "threshold_opt" : round(m["best_threshold"], 3),
            "FN": m["FN"], "FP": m["FP"], "TN": m["TN"], "TP": m["TP"],
            "fn_rate"       : round(m["fn_rate"], 4),
        })

        final = fit_final_model(X, y, xgb_params)
        mlflow.xgboost.log_model(final["clf"], artifact_path="model")
        run_id = run.info.run_id

    sep = "-" * 60
    print("Run:", run_name, "| ID:", run_id[:8] + "...")
    print(sep)
    print("  AUC-OOF    :", round(m["auc_oof"], 4),
          "  gap =", round(m["gap"], 3))
    print("  PR-AUC     :", round(m["pr_auc"], 4),
          "  (chance =", round(y.mean(), 3), ")")
    print("  F1         :", round(m["f1"], 4),
          "  thr =", round(m["best_threshold"], 3))
    print("  Sensitivity:", round(m["sensitivity"], 4),
          "  Specificity:", round(m["specificity"], 4))
    print("  CM: TN=" + str(m["TN"]) + " FP=" + str(m["FP"]) +
          " FN=" + str(m["FN"]) + " TP=" + str(m["TP"]))
    print(sep)

    return {
        "run_id"      : run_id, "run_name": run_name,
        "metrics"     : m, "oof_probs": cv_out["oof_probs"],
        "feature_cols": used_cols, "final_model": final,
    }


print("run_experiment() ready — TAP K=2 binary")


## 8. Phase 1 Experiments — Birth → 3 Months CA

| Run | Features | Params | Purpose |
|-----|----------|--------|---------|
| `P1_full` | ALL_FEATURES_P1 | XGB_PARAMS | full |
| `P1_reg` | ALL_FEATURES_P1 | XGB_PARAMS_REG | regularised |
| `P1_spw` | ALL_FEATURES_P1 | XGB_PARAMS_SPW | spw |


In [0]:
results_p1 = {}
header = "=" * 60

print(header)
print("PHASE 1 — Birth to 3 months CA  |  TAP K=2")
print(header)

print()
print("Run 1/3 — P1_full (full)")
results_p1["P1_full"] = run_experiment(
    run_name     = "P1_full",
    feature_cols = ALL_FEATURES_P1,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "1", "experiment_type": "full"},
)

print()
print("Run 2/3 — P1_reg (regularised)")
results_p1["P1_reg"] = run_experiment(
    run_name     = "P1_reg",
    feature_cols = ALL_FEATURES_P1,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REG,
    tags         = {"phase": "1", "experiment_type": "regularised"},
)

print()
print("Run 3/3 — P1_spw (spw)")
results_p1["P1_spw"] = run_experiment(
    run_name     = "P1_spw",
    feature_cols = ALL_FEATURES_P1,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_SPW,
    tags         = {"phase": "1", "experiment_type": "spw"},
)

print()
print("Phase 1 complete.")

## 9. Phase 2 Experiments — Birth → 12 Months CA

| Run | Features | Params | Purpose |
|-----|----------|--------|---------|
| `P2_full` | ALL_FEATURES_P2 | XGB_PARAMS | full |
| `P2_reg` | ALL_FEATURES_P2 | XGB_PARAMS_REG | regularised |
| `P2_spw` | ALL_FEATURES_P2 | XGB_PARAMS_SPW | spw |
| `P2_top15_reg` | FEATURES_TOP15 | XGB_PARAMS_REG | top15_regularised |


In [0]:
results_p2 = {}

print(header)
print("PHASE 2 — Birth to 12 months CA  |  TAP K=2")
print(header)

FEATURES_TOP15 = [
    "F_delta_waz_3m_12m", "SCB_nivm1", "F_catchup_hc_fenton",
    "PMD_coaudl6", "PMD_RSM6", "SCB_percap1", "EX41_talla8",
    "NEO_fotote6", "PMD_coloco12", "NEO_totoxidias",
    "F_z_hc_birth_fenton", "F_delta_haz_3m_12m", "SCB_nivp1",
    "NEO_HOSP", "PMD_cogrif6",
]

print()
print("Run 1/4 — P2_full (full)")
results_p2["P2_full"] = run_experiment(
    run_name     = "P2_full",
    feature_cols = ALL_FEATURES_P2,
    df           = df_g1,
    xgb_params   = XGB_PARAMS,
    tags         = {"phase": "2", "experiment_type": "full"},
)

print()
print("Run 2/4 — P2_reg (regularised)")
results_p2["P2_reg"] = run_experiment(
    run_name     = "P2_reg",
    feature_cols = ALL_FEATURES_P2,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REG,
    tags         = {"phase": "2", "experiment_type": "regularised"},
)

print()
print("Run 3/4 — P2_spw (spw)")
results_p2["P2_spw"] = run_experiment(
    run_name     = "P2_spw",
    feature_cols = ALL_FEATURES_P2,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_SPW,
    tags         = {"phase": "2", "experiment_type": "spw"},
)

print()
print("Run 4/4 — P2_top15_reg (top15_regularised)")
results_p2["P2_top15_reg"] = run_experiment(
    run_name     = "P2_top15_reg",
    feature_cols = FEATURES_TOP15,
    df           = df_g1,
    xgb_params   = XGB_PARAMS_REG,
    tags         = {"phase": "2", "experiment_type": "top15_regularised"},
)

print()
print("Phase 2 complete.")

## 10. Cross-Phase Comparison

In [0]:
all_results = {**results_p1, **results_p2}

rows = []
for name, res in all_results.items():
    m = res["metrics"]
    rows.append({
        "Run"         : name,
        "Phase"       : "1" if name.startswith("P1") else "2",
        "N_features"  : len(res["feature_cols"]),
        "AUC_OOF"     : round(m["auc_oof"], 4),
        "PR_AUC"      : round(m["pr_auc"], 4),
        "Gap"         : round(m["gap"], 4),
        "F1"          : round(m["f1"], 4),
        "Sensitivity" : round(m["sensitivity"], 4),
        "Specificity" : round(m["specificity"], 4),
        "FN"          : m["FN"],
        "FP"          : m["FP"],
        "Threshold"   : round(m["best_threshold"], 3),
    })

df_compare = pd.DataFrame(rows).sort_values("AUC_OOF", ascending=False)
print("Cross-phase comparison — TAP K=2")
print(df_compare.to_string(index=False))

best_name = df_compare.iloc[0]["Run"]
best      = all_results[best_name]
print()
print("Best run:", best_name)

# AUC bar chart
fig, ax = plt.subplots(figsize=(10, 4))
colors = [PAL["dark_blue"] if r.startswith("P2") else PAL["mid_blue"]
          for r in df_compare["Run"]]
bars = ax.bar(df_compare["Run"], df_compare["AUC_OOF"],
              color=colors, edgecolor="white", alpha=0.88)
ax.axhline(0.5,  color="red",          ls="--", lw=1.2, alpha=0.5, label="Chance")
ax.axhline(0.678, color=PAL["green"], ls=":",  lw=1.5, alpha=0.7,
           label="M7 reference (0.678)")
ax.set_ylim(0.40, 0.82)
ax.set_ylabel("AUC-ROC (OOF)", fontsize=11)
ax.set_title(
    "M9 TAP K=2 — AUC by experiment\n"
    "Saving Brains · GO-2 Inattentive prediction · G1 preterm",
    fontweight="bold", fontsize=11,
)
ax.tick_params(axis="x", labelsize=8, rotation=15)
ax.legend(fontsize=9)
for bar, auc in zip(bars, df_compare["AUC_OOF"]):
    ax.text(bar.get_x() + bar.get_width()/2, auc + 0.007,
            f"{auc:.3f}", ha="center", fontsize=8.5, fontweight="bold")
from matplotlib.patches import Patch
legend_els = [Patch(color=PAL["dark_blue"], label="Phase 2 (12m CA)"),
              Patch(color=PAL["mid_blue"],  label="Phase 1 (3m CA)")]
ax.legend(handles=legend_els + ax.get_legend_handles_labels()[0][2:], fontsize=8)
plt.tight_layout()
plt.show()


## 11. Best Model Analysis — Metrics + ROC + SHAP

In [0]:
import shap
import matplotlib.gridspec as gridspec

m   = best["metrics"]
oof = best["oof_probs"]
X_b, y_b, feat_b = prepare_arrays(df_g1, best["feature_cols"])
thr = m["best_threshold"]
pred = (oof >= thr).astype(int)
tn, fp, fn, tp = m["TN"], m["FP"], m["FN"], m["TP"]
auc_str = str(round(m["auc_oof"], 4))
thr_str = str(round(thr, 3))

print("Best model:", best["run_name"])
print("AUC:", auc_str, " F1:", round(m["f1"],4),
      " thr:", thr_str)

# ── Metrics panel ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

ax1 = fig.add_subplot(gs[0])
m_n = ["AUC-ROC","PR-AUC","F1-Score","Sensitivity","Specificity","NPV"]
m_v = [m["auc_oof"],m["pr_auc"],m["f1"],m["sensitivity"],m["specificity"],m["npv"]]
m_c = [PAL["dark_blue"],PAL["purple"],PAL["mid_blue"],PAL["green"],PAL["red"],PAL["orange"]]
bars = ax1.barh(m_n, m_v, color=m_c, edgecolor="white", alpha=0.88)
ax1.set_xlim(0,1.14); ax1.axvline(0.5,color="grey",ls="--",lw=1,alpha=0.5)
for bar, val in zip(bars, m_v):
    ax1.text(val+0.02, bar.get_y()+bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=9.5, fontweight="bold")
ax1.set_title("Metrics — " + best["run_name"] + "\n(thr=" + thr_str + ")",
              fontweight="bold", fontsize=11)

ax2 = fig.add_subplot(gs[1])
cm2 = np.array([[tn, fp],[fn, tp]])
ax2.imshow(cm2, cmap="Blues", vmin=0, vmax=cm2.max()*1.3)
labels = [["TN","FP"],["FN","TP"]]
for i in range(2):
    for j in range(2):
        v = cm2[i,j]
        col = "white" if v > cm2.max()*0.55 else "black"
        ax2.text(j, i, labels[i][j]+"\n"+str(v),
                 ha="center", va="center", fontsize=13,
                 fontweight="bold", color=col)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(["Pred GO-1","Pred GO-2"], fontsize=10)
ax2.set_yticklabels(["True GO-1","True GO-2"], fontsize=10)
ax2.set_title("Confusion Matrix\nFN=" + str(fn) + "  FP=" + str(fp),
              fontweight="bold", fontsize=11)

ax3 = fig.add_subplot(gs[2])
fpr, tpr, _ = roc_curve(y_b, oof)
ax3.plot(fpr, tpr, lw=2.5, color=PAL["dark_blue"],
         label=best["run_name"] + " (AUC=" + auc_str + ")")
ax3.plot([0,1],[0,1],"k--",lw=1,alpha=0.4,label="Chance")
ax3.set_xlabel("1 - Specificity (FPR)", fontsize=10)
ax3.set_ylabel("Sensitivity (TPR)",     fontsize=10)
ax3.set_title("ROC Curve", fontweight="bold", fontsize=11)
ax3.legend(fontsize=8, loc="lower right")

fig.suptitle("Best model analysis — " + best["run_name"],
             fontweight="bold", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# ── SHAP ──────────────────────────────────────────────────────────────────
final_md = best["final_model"]
clf      = final_md["clf"]
imp_f    = final_md["imputer"]
sc_f     = final_md["scaler"]
cols_ok  = [c for c in best["feature_cols"] if c in df_g1.columns]

to_n    = lambda s: pd.to_numeric(s, errors="coerce")
X_raw   = df_g1[cols_ok].apply(to_n).values.astype(float)
X_imp   = imp_f.transform(X_raw)
X_sc    = sc_f.transform(X_imp)

N_SHAP  = min(180, X_sc.shape[0])
shap_idx= np.random.RandomState(RANDOM_STATE).choice(X_sc.shape[0],N_SHAP,replace=False)
explainer   = shap.TreeExplainer(clf)
shap_values = explainer(X_sc[shap_idx], check_additivity=False)
shap_values.feature_names = cols_ok

mean_shap = np.abs(shap_values.values).mean(axis=0)
df_shap   = (pd.DataFrame({"feature": cols_ok, "shap_mean": mean_shap})
             .sort_values("shap_mean", ascending=False).reset_index(drop=True))
df_shap["rank"] = range(1, len(df_shap)+1)

print("\nTop 10 SHAP features:")
print(df_shap.head(10).to_string(index=False))

fig, (ax_bee, ax_bar) = plt.subplots(1, 2, figsize=(14, 6))
plt.sca(ax_bee)
shap.plots.beeswarm(shap_values, max_display=12, show=False)
ax_bee.set_title("SHAP Beeswarm — " + best["run_name"], fontweight="bold", fontsize=11)

top15 = df_shap.head(15).sort_values("shap_mean")
ax_bar.barh(top15["feature"], top15["shap_mean"],
            color=PAL["dark_blue"], edgecolor="white", alpha=0.88)
ax_bar.set_xlabel("Mean |SHAP| value", fontsize=11)
ax_bar.set_title("Top-15 features — SHAP importance",
                 fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()


## 12. M9 vs M7 — Direct Comparison

In [0]:
# ─────────────────────────────────────────────────────────────────────────
# Compare M9 (TAP K=2) vs M7 (binary GO-i K=2, composite phenotype)
# TAP phenotype (attention/WM) is clinically orthogonal to
# the WASI+CVLT composite GO-i. Compare with M7 as reference only.
# Lower TAP WM omissions → better attention → lower risk.
# ─────────────────────────────────────────────────────────────────────────

m7_ref = {
    "AUC_OOF"     : 0.678,
    "PR_AUC"      : 0.468,
    "F1"          : 0.497,
    "Sensitivity" : 0.779,
    "Specificity" : 0.524,
    "FN"          : 25,
    "FP"          : 127,
    "Gap"         : 0.157,
}

m_best = best["metrics"]
print("=" * 68)
print("  M9 (TAP K=2)  vs  M7 (GO-i K=2 composite)")
print("=" * 68)
print(f"  {'Metric':<22} {'M7 ref':>10}  {'M9 best':>10}  {'Delta':>10}")
print("  " + "─" * 60)

pairs = [
    ("AUC-OOF",     m7_ref["AUC_OOF"],     round(m_best["auc_oof"],4)),
    ("PR-AUC",      m7_ref["PR_AUC"],       round(m_best["pr_auc"],4)),
    ("F1-Score",    m7_ref["F1"],           round(m_best["f1"],4)),
    ("Sensitivity", m7_ref["Sensitivity"],  round(m_best["sensitivity"],4)),
    ("Specificity", m7_ref["Specificity"],  round(m_best["specificity"],4)),
    ("FN (missed)", m7_ref["FN"],           m_best["FN"]),
    ("FP (alarms)", m7_ref["FP"],           m_best["FP"]),
    ("Gap",         m7_ref["Gap"],          round(m_best["gap"],4)),
]
for name, v7, vm in pairs:
    delta = round(vm - v7, 4) if isinstance(vm, float) else vm - v7
    arrow = "▲" if delta > 0 else "▼" if delta < 0 else "─"
    print(f"  {name:<22} {str(v7):>10}  {str(vm):>10}  {arrow} {delta}")

print()
print("Interpretation:")
print("  TAP phenotype vs GO-i composite:")
print("  TAP phenotype (attention/WM) is orthogonal to GO-i composite.")
print("  Compare with M7 as reference only.")
print("  Lower TAP WM omissions = better attention = lower risk.")


## 13. Save Artefacts & Final Summary

In [0]:
import joblib

df_compare.to_csv("m9_tap_resultados.csv", index=False, encoding="utf-8-sig")
df_shap.to_csv("m9_tap_shap.csv",          index=False, encoding="utf-8-sig")
df_screen.to_csv("m9_tap_screening.csv",   index=False, encoding="utf-8-sig")

bundle = {
    "clf"          : best["final_model"]["clf"],
    "imputer"      : best["final_model"]["imputer"],
    "scaler"       : best["final_model"]["scaler"],
    "feature_cols" : best["feature_cols"],
    "threshold"    : best["metrics"]["best_threshold"],
    "metrics"      : best["metrics"],
    "run_id"       : best["run_id"],
    "model"        : "M9_TAP_K2",
    "target"       : "GO-2 Inattentive = 1",
}
joblib.dump(bundle, "m9_tap_bundle.joblib")

print("Saved:")
print("  m9_tap_resultados.csv")
print("  m9_tap_shap.csv")
print("  m9_tap_screening.csv")
print("  m9_tap_bundle.joblib")
print()

sep = "=" * 62
print(sep)
print("  FINAL SUMMARY — M9 TAP K=2")
print(sep)
m_b = best["metrics"]
print("  Best run      :", best["run_name"])
print("  AUC-OOF       :", round(m_b["auc_oof"],4))
print("  PR-AUC        :", round(m_b["pr_auc"],4))
print("  F1            :", round(m_b["f1"],4),
      "  thr =", round(m_b["best_threshold"],3))
print("  Sensitivity   :", round(m_b["sensitivity"],4))
print("  Specificity   :", round(m_b["specificity"],4))
print("  FN            :", m_b["FN"], "(GO-2 Inattentive missed)")
print("  Gap           :", round(m_b["gap"],4))
print(sep)
print()
uri_str = mlflow.get_tracking_uri()
print("MLflow URI:", uri_str)
print("Launch UI : mlflow ui --backend-store-uri " + uri_str)
